In [2]:
import torch
from torch import nn
from datasets import load_dataset
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [69]:
from datasets import Dataset as dts

In [42]:
from nltk.corpus import stopwords

In [73]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.utils import resample

In [16]:
from json import loads

In [3]:
dataset = load_dataset('s-nlp/ru_non_detoxified')

In [61]:
ds = dataset['train']


In [56]:
ds['reasons']

['{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"not_toxic":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"unclear":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"not_toxic":true}',
 '{"toxic_content":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"unclear":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"not_toxic":true}',
 '{"not_toxic":true}',
 '{"toxic_content":true}',
 '{"toxic_content":true}',
 '{"not_

In [30]:
def label_encoding(d):
    try: 
        data = loads(d)
        if data.get('not_toxic') is True:
            return 0
        if data.get('toxic_content') is True:
            return 1
        else:
            return None
    except Exception:
        return None



In [ ]:
ds = ds.to_pandas()
ds['label'] = ds['reasons'].apply(label_encoding)

In [63]:
ds.dropna(inplace=True)
ds.drop(columns=['reasons'], inplace=True)

In [75]:
major_class = ds[ds['label']== 1]
minor_class = ds[ds['label'] ==0]

In [78]:
ds_upsampled = resample(
    minor_class, replace=True, n_samples=len(major_class), random_state=42
)

In [80]:
ds = pd.concat([major_class, ds_upsampled])

In [83]:
ds.drop(columns=['__index_level_0__'], inplace=True)

In [84]:
ds = dts.from_pandas(ds)

In [85]:
X_train, X_test, y_train, y_test = train_test_split(
    ds['toxic_comment'], ds['label'], test_size=0.2, stratify=ds['label'], random_state=42
)

In [91]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=0.15, shuffle=True, stratify=y_train
)

In [98]:
class BoWDS(Dataset):
    def __init__(self, embeds, labels):
        self.embeds = torch.FloatTensor(embeds)
        self.labels = torch.FloatTensor(labels)


    def __getitem__(self, index):

        return {
            'embed': self.embeds[index],
            'label': self.labels[index]



        } 
    def __len__(self):
        return len(self.embeds)



## BoW train

In [ ]:
stop_words = stopwords.words('russian')


In [47]:
vectorizer = CountVectorizer(
    min_df=5, max_df=0.95, max_features=8000,
    binary=True, stop_words=stop_words
)

In [93]:
vectorizer.fit(X_train)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.","['и', 'в', ...]"
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


In [94]:
X_train = vectorizer.transform(X_train).toarray()

In [104]:
X_val = vectorizer.transform(X_val).toarray()
X_test = vectorizer.transform(X_test).toarray()

In [99]:
train_ds = BoWDS(X_train, y_train)

In [105]:
val_ds = BoWDS(X_val, y_val)

In [106]:
train_dataloader= DataLoader(train_ds, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_ds, batch_size=62, shuffle=True)

In [107]:
s = next(iter(train_dataloader))

In [ ]:
inp_dim = len(vectorizer.vocabulary_)

6224

In [ ]:
class Mymodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(inp_dim, 1)





torch.Size([64, 6224])